In [2]:
from ultralytics import YOLO
import numpy as np
import cv2
import torch
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
import timm
import torch.optim as optim
import os
from datetime import datetime

/home/hice1/axu39/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
NUM_LANDMARKS = 9
INPUT_SIZE = 512


class HRNetLandmarkModel(nn.Module):
    def __init__(self, num_landmarks=NUM_LANDMARKS, pretrained=True, num_heads=8):
        super().__init__()

        self.backbone = timm.create_model(
            "hrnet_w18",
            pretrained=pretrained,
            features_only=True
        )

        self.feature_dim = self.backbone.feature_info.channels()[-1]
        self.num_landmarks = num_landmarks

        # landmark queries
        self.landmark_queries = nn.Parameter(
            torch.randn(num_landmarks, self.feature_dim) * 0.02
        )

        # cross attention (queries attend to image tokens)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=self.feature_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )

        # self attention between landmarks
        self.self_attn = nn.MultiheadAttention(
            embed_dim=self.feature_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )

        # coordinate head
        self.coord_head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Linear(self.feature_dim, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        feats = self.backbone(x)[-1]  # B C H W
        B, C, H, W = feats.shape

        tokens = feats.flatten(2).transpose(1, 2)  # B HW C

        queries = self.landmark_queries.unsqueeze(0).expand(B, -1, -1)

        queries, _ = self.cross_attn(
            queries,
            tokens,
            tokens
        )

        queries, _ = self.self_attn(
            queries,
            queries,
            queries
        )

        coords = torch.sigmoid(self.coord_head(queries))

        return coords

In [20]:
class LizardDataset(Dataset):
    def __init__(self, pt_paths, input_size=512, heatmap_size=512):
        self.paths = pt_paths
        self.input_size = input_size
        self.heatmap_size = heatmap_size

        self.transform = A.Compose([
            A.LongestMaxSize(max_size=input_size),
            A.PadIfNeeded(input_size, input_size, border_mode=cv2.BORDER_CONSTANT),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.1,
                rotate_limit=25,
                border_mode=cv2.BORDER_REFLECT_101,
                p=0.8
            ),
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10)
            ], p=0.7),
            A.ElasticTransform(alpha=10, sigma=5, p=0.2),
            A.GaussNoise(var_limit=(1,5), p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
        ], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        data = torch.load(self.paths[idx])
        img = data["image"].permute(1,2,0).numpy()
        coords = data["tps"].numpy()  # shape (9,2)

        # Clip coords inside original image bounds
        H, W = img.shape[:2]
        coords[:,0] = np.clip(coords[:,0], 0, W-1)
        coords[:,1] = np.clip(coords[:,1], 0, H-1)
        keypoints = coords.tolist()

        augmented = self.transform(image=img, keypoints=keypoints)
        img_aug = augmented["image"]
        kp_aug = np.array(augmented["keypoints"], dtype=np.float32)

        if kp_aug.shape[0] != 9:
            raise ValueError(f"Augmented keypoints shape mismatch: {kp_aug.shape}, expected 9")

        # Clamp and normalize
        kp_aug[:,0] = np.clip(kp_aug[:,0], 0, self.input_size-1)
        kp_aug[:,1] = np.clip(kp_aug[:,1], 0, self.input_size-1)
        coords_norm = kp_aug / self.input_size

        img_tensor = torch.from_numpy(img_aug).permute(2,0,1).float()
        coords_tensor = torch.from_numpy(coords_norm).float()

        return img_tensor, coords_tensor

In [21]:
pt_files = [os.path.join(DATA_PATH,f) for f in os.listdir(DATA_PATH) if f.endswith(".pt")]

dataset = LizardDataset(pt_files, input_size=INPUT_SIZE)

val_len = int(len(dataset) * VALIDATION_FRACTION)
train_len = len(dataset) - val_len

train_dataset, val_dataset = random_split(dataset,[train_len,val_len])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


model = HRNetLandmarkModel().to(DEVICE)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': LR_BACKBONE},
    {'params': model.cross_attn.parameters(), 'lr': LR_HEAD},
    {'params': model.self_attn.parameters(), 'lr': LR_HEAD},
    {'params': model.coord_head.parameters(), 'lr': LR_HEAD},
    {'params': [model.landmark_queries], 'lr': LR_HEAD}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.3,
    patience=5
)

criterion = torch.nn.SmoothL1Loss()


def denormalize(img):

    mean = np.array([0.485,0.456,0.406])
    std = np.array([0.229,0.224,0.225])

    img = img * std + mean
    img = np.clip(img * 255,0,255).astype(np.uint8)

    return img


def overlay_landmarks(image_tensor, coords, save_path):

    img = image_tensor.permute(1,2,0).cpu().numpy()
    img = denormalize(img)

    H,W,_ = img.shape

    coords = coords.cpu().numpy()

    for i,(x,y) in enumerate(coords):

        px = int(x*W)
        py = int(y*H)

        cv2.circle(img,(px,py),6,(0,255,0),-1)

        cv2.putText(
            img,
            str(i),
            (px+4,py-4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (255,255,0),
            1
        )

    cv2.imwrite(save_path,img)


for epoch in range(1,EPOCHS+1):

    model.train()
    train_loss = 0

    for imgs, coords_gt in train_loader:

        imgs = imgs.to(DEVICE)
        coords_gt = coords_gt.to(DEVICE)

        optimizer.zero_grad()

        coords_pred = model(imgs)

        loss = criterion(coords_pred, coords_gt)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():

        for batch_idx,(imgs,coords_gt) in enumerate(val_loader):

            imgs = imgs.to(DEVICE)
            coords_gt = coords_gt.to(DEVICE)

            coords_pred = model(imgs)

            loss = criterion(coords_pred, coords_gt)

            val_loss += loss.item()

            if batch_idx == 0:

                overlay_landmarks(
                    imgs[0],
                    coords_pred[0],
                    os.path.join(OVERLAY_DIR,f"epoch{epoch}.png")
                )

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch} | "
        f"Train {avg_train_loss:.4f} | "
        f"Val {avg_val_loss:.4f}"
    )

    scheduler.step(avg_val_loss)

/tmp/ipykernel_1351654/4152106250.py:22: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(1,5), p=0.3),
Unexpected keys (downsamp_modules.0.1.num_batches_tracked, downsamp_modules.1.1.num_batches_tracked, downsamp_modules.2.1.num_batches_tracked, final_layer.1.num_batches_tracked, downsamp_modules.0.0.bias, downsamp_modules.0.0.weight, downsamp_modules.0.1.bias, downsamp_modules.0.1.running_mean, downsamp_modules.0.1.running_var, downsamp_modules.0.1.weight, downsamp_modules.1.0.bias, downsamp_modules.1.0.weight, downsamp_modules.1.1.bias, downsamp_modules.1.1.running_mean, downsamp_modules.1.1.running_var, downsamp_modules.1.1.weight, downsamp_modules.2.0.bias, downsamp_modules.2.0.weight, downsamp_modules.2.1.bias, downsamp_modules.2.1.running_mean, downsamp_modules.2.1.running_var, downsamp_modules.2.1.weight, final_layer.0.bias, final_layer.0.weight, final_layer.1.bias, final_layer.1.running_mean, final_layer.1.running_var, final

ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/hice1/axu39/scratch/shg/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/hice1/axu39/scratch/shg/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
  File "/home/hice1/axu39/scratch/shg/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 416, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/home/hice1/axu39/scratch/shg/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 416, in <listcomp>
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/tmp/ipykernel_1351654/4152106250.py", line 45, in __getitem__
    raise ValueError(f"Augmented keypoints shape mismatch: {kp_aug.shape}